# Academic Report Summarizer (BART-large-CNN, GPU)
Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.
Uses `model.generate()` directly, so it works with old and new `transformers` versions.

In [ ]:
!pip install -q transformers pypdf
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"   # ~1.6 GB download
CHUNK_TOKENS = 900                        # under BART's 1024-token input limit
INPUT_LIMIT = 1024

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if device == "cuda":
    model = model.half()          # fp16: faster and lighter on GPU
model.to(device).eval()
print("Model loaded on", device.upper())

In [ ]:
def clean_text(text):
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)               # fix hyphenated line breaks
    m = re.search(r"\n\s*(references|bibliography)\s*\n", text, re.IGNORECASE)
    if m and m.start() > len(text) * 0.5:                        # drop trailing reference list
        text = text[:m.start()]
    return re.sub(r"\s+", " ", text).strip()

def read_report(path):
    if path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        text = "\n".join((pg.extract_text() or "") for pg in PdfReader(path).pages)
    else:
        text = open(path, encoding="utf-8", errors="ignore").read()
    return clean_text(text)

def n_tok(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def split_into_chunks(text, max_tokens=CHUNK_TOKENS):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, cur, cur_len = [], [], 0
    for s in sentences:
        n = n_tok(s)
        if n > max_tokens:                                       # oversized "sentence": hard split
            if cur:
                chunks.append(" ".join(cur)); cur, cur_len = [], 0
            ids = tokenizer.encode(s, add_special_tokens=False)
            for i in range(0, len(ids), max_tokens):
                chunks.append(tokenizer.decode(ids[i:i + max_tokens]))
            continue
        if cur_len + n > max_tokens:
            chunks.append(" ".join(cur)); cur, cur_len = [], 0
        cur.append(s); cur_len += n
    if cur:
        chunks.append(" ".join(cur))
    return chunks

def summarize_once(text, max_len, min_len):
    n = n_tok(text)
    max_len = max(20, min(max_len, int(n * 0.7)))                # keep output shorter than input
    min_len = max(10, min(min_len, max_len - 5))
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=INPUT_LIMIT).to(device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=max_len,
            min_length=min_len,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

def summarize_report(text, max_length=150, min_length=50):
    total = n_tok(text)
    if total < 60:
        return text
    level = 0
    while total > CHUNK_TOKENS:                                  # map: summarize chunks repeatedly
        level += 1
        chunks = split_into_chunks(text)
        print(f"Pass {level}: {len(chunks)} chunks")
        text = " ".join(summarize_once(c, 120, 40) for c in chunks)
        new_total = n_tok(text)
        if new_total >= total:                                   # safety: no progress
            break
        total = new_total
    return summarize_once(text, max_length, min_length)          # reduce: final summary

## Load your report
Upload a `.pdf` or `.txt` file (or skip and paste text in the note below).

In [ ]:
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))
report_text = read_report(path)
print(f"Loaded '{path}': {len(report_text):,} characters, {n_tok(report_text):,} tokens")

In [ ]:
summary = summarize_report(report_text, max_length=150, min_length=50)
print("=== SUMMARY ===\n")
print(summary)

with open("summary.txt", "w") as f:
    f.write(summary)
# files.download("summary.txt")   # uncomment to download

### Optional: paste text instead of uploading
```python
report_text = clean_text("""paste your report here""")
```